In [9]:
##modules
#%matplotlib widget
%matplotlib inline
#
#%matplotlib qt
import mne
import numpy as np
import matplotlib
# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.

import matplotlib.pyplot as plt

#matplotlib.use('TkAgg')  # Asegúrate de que este backend está instalado.

import pandas as pd 
import os
import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

# aplicar la acf EN epochs

from statsmodels.tsa.stattools import acf


from scipy.stats import permutation_test
sys.path.append(r"G:\apuntes_mne\code_MOUS\scripts_functions_created")

# Ahora importa la función
from print5 import print5


import pickle

from scipy.stats import gaussian_kde
import numpy as np
import matplotlib.pyplot as plt

# Import required code for visualizing example models
from fooof import FOOOF
from fooof.sim.gen import gen_power_spectrum
from fooof.sim.utils import set_random_seed
from fooof.plts.spectra import plot_spectra
from fooof.plts.annotate import plot_annotated_model
from fooof import FOOOFGroup,Bands
from fooof.analysis.periodic import get_band_peak_group, get_band_peak


In [10]:

try:
    # Si se ejecuta como SCRIPT .py: usar __file__
    sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
except NameError:
    # Si se ejecuta como NOTEBOOK Jupyter: usar path relativo
    sys.path.append("..")  # sube un nivel desde la carpeta actual del notebook


# --- Configuración dinámica de rutas ---
from get_paths_SELF import get_paths_SELF

# Parámetros editables
disco = "g"
modality="visual"
layer_script = "event"
subj= "s01b"
type_epoch="emoc"

# Generar variables automáticamente
path_dict = get_paths_SELF(disco=disco, modality=modality,layer_script=layer_script,  subj=subj)
globals().update(path_dict)

# Mostrar todos los paths generados
print("\n📁 Rutas generadas:")
for k, v in path_dict.items():
    print(f"{k:<20} → {v}")

    


✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\epochs_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\ICA_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\epochs_clean_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\epochs_matlab_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\evoked_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\channels_structure
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_source\source_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_source\source_event\raw_hsp
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_source\source_event\fwd
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_source\source_event\inverse
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_analysis\analysis_event\acw_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_analysis\analysis_eve

In [11]:
subjects_BV = []

# Define modality pattern
if modality == "visual":
    pattern_modality = "_vis_"
elif modality == "auditory":
    pattern_modality = "_aud_"
else:
    pattern_modality = None

# Read subject names from .vhdr files in export_generic_data
for archivo in export_generic_data.glob("*.vhdr"):
    nombre = archivo.stem  # filename without extension

    # Keep only files matching the selected modality
    if pattern_modality is not None and pattern_modality in nombre:
        sujeto = nombre.split("_")[0].lower()  # e.g. s01b_vis_c_BV_mne -> s01b
        subjects_BV.append(sujeto)

# Remove duplicates and sort ignoring case
subjects_BV = sorted(set(subjects_BV), key=str.lower)

print("Subjects found:")
print(subjects_BV)




# --------------------------------------------------
# Channels: read them from the CSV created before
# --------------------------------------------------
channels = pd.read_csv(channels_structure_path / f"channels_{modality}.csv")

# Keep EEG channels in original order
channels_eeg = channels.loc[channels["type"] == "eeg", "channel"].tolist()

# Optional: get EOG channels too
channels_eog = channels.loc[channels["type"] == "eog", "channel"].tolist()

print("\nEEG channels:")
print(channels_eeg)

print("\nEOG channels:")
print(channels_eog)

del channels

filtering=True
if filtering==True:
    lfreq=1
    hfreq=40
    filter_name= f"filt_{lfreq}-{hfreq}"
    print(f"Filtrado aplicado: {lfreq}-{hfreq} Hz")
elif filtering==False:
    print("No se ha aplicado filtrado.")


Subjects found:
['s01b', 's02b', 's03b', 's04b', 's05b', 's06b', 's07b', 's08b', 's09b', 's10b', 's11b', 's12b', 's13b', 's14b', 's15b', 's16b', 's17b', 's18b', 's19b', 's20b', 's21b', 's22b', 's23b', 's24b', 's25b', 's26b', 's27b', 's28b', 's29b', 's30b', 's31b', 's32b', 's33b', 's34b', 's35b', 's36b']

EEG channels:
['Fp1', 'Fpz', 'Fp2', 'AF7', 'AF3', 'AF4', 'AF8', 'F7', 'F5', 'F3', 'F1', 'Fz', 'F2', 'F4', 'F6', 'F8', 'FT7', 'FC5', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4', 'FC6', 'FT8', 'T7', 'C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6', 'T8', 'TP7', 'CP5', 'CP3', 'CP1', 'CPz', 'CP2', 'CP4', 'CP6', 'TP8', 'P7', 'P5', 'P3', 'P1', 'Pz', 'P2', 'P4', 'P6', 'P8', 'PO7', 'PO3', 'PO4', 'PO8', 'O1', 'Oz', 'O2']

EOG channels:
['HEOG+', 'HEOG', 'VEOG+', 'VEOG']
Filtrado aplicado: 1-40 Hz


In [12]:
def check_nans(data, nan_policy='zero'):
    """Check an array for nan values, and replace, based on policy."""

    # Find where there are nan values in the data
    nan_inds = np.where(np.isnan(data))

    # Apply desired nan policy to data
    if nan_policy == 'zero':
        data[nan_inds] = 0
    elif nan_policy == 'mean':
        data[nan_inds] = np.nanmean(data)
    else:
        raise ValueError('Nan policy not understood.')

    return data

In [13]:


def compute_fooof_subject(subject,epochs, bands,change_name=None ,aperiodic_mode="fixed", select_highest=False, select_channels=None,return_fg_subject=False, picks="eeg" ):

    #list to append the data frames 
    list_df = []
    #dictionary to append the fooof groups
    dict_fg={}
    
    epochs = epochs.copy().pick(picks=picks, exclude="bads")
    
    #now 
    condition_names = list(epochs.event_id.keys())

    
    #but i will keep the original names to select the epochs
    for i_cond in range(0,len(condition_names)):
        
        #now i take the original name to select the epochs
        cond=condition_names[i_cond]
    
        #now i select the epochs for that condition
        epochs_cond = epochs[cond]
        
        #if any condition has no epochs i skip it
        if len(epochs_cond) == 0:
            continue
    
        #if change name i chanege the name of the condition
        if change_name:
            cond_clean=cond.removeprefix(f"{change_name}_")
        else:
            cond_clean=cond



        
        channels = epochs_cond.ch_names
        data = epochs_cond.get_data()   # (n_epochs, n_channels, n_times)
        sfreq = epochs_cond.info['sfreq']
        fmin=epochs_cond.info['highpass']
        fmax=epochs_cond.info['lowpass']

        n_epochs, n_channels, n_times = data.shape
        
         # Event codes and condition labels for the selected epochs
        epoch_codes = epochs_cond.events[:, 2]

        Condition_self = []
        Condition_emotion = []
        Condition_gaze = []

        for code in epoch_codes:
            code_str = str(code).zfill(2)   # ensure 2-digit format
            first_digit = int(code_str[0])  # self / gaze information
            second_digit = int(code_str[1]) # emotion information

            # SELF condition
            if first_digit in [1, 2, 3]:
                self_label = "self"
            elif first_digit in [4, 5, 6]:
                self_label = "friend"
            elif first_digit in [7, 8, 9]:
                self_label = "unknown"
            else:
                self_label = np.nan

            # GAZE direction
            if first_digit in [1, 4, 7]:
                gaze_label = 1
            elif first_digit in [2, 5, 8]:
                gaze_label = 2
            elif first_digit in [3, 6, 9]:
                gaze_label = 3
            else:
                gaze_label = np.nan

            # EMOTION condition
            if second_digit == 4:
                emotion_label = "positive"
            elif second_digit == 5:
                emotion_label = "neutral"
            elif second_digit == 6:
                emotion_label = "negative"
            else:
                emotion_label = np.nan

            Condition_self.append(self_label)
            Condition_emotion.append(emotion_label)
            Condition_gaze.append(gaze_label)

        # Número total de modelos (n_epochs * n_channels)
        shape_tabla = n_epochs * n_channels
        print(f"Number of epochs: {n_epochs}, Number of channels: {n_channels},")

        # Parámetros PSD
        #n_per_seg = el número de muestras por segmento que se usa para calcular una ventana del método de Welch.

        #lower oscillation is the inverse of the lowest frequency, multiplied by 3 to ensure at least 3 cycles are captured
        lower_oscillation=(1/fmin)*3
        n_per_seg = int(lower_oscillation * sfreq)

        psd, freqs = mne.time_frequency.psd_array_welch(
            data,
            sfreq=sfreq,
            fmin=fmin,
            fmax=fmax,
            n_per_seg=n_per_seg,
            # n_fft=n_per_seg,
            n_jobs=20
        )

        # psd: (n_epochs, n_channels, n_freqs)
        psd_flat = psd.reshape(-1, psd.shape[-1])   # (n_epochs * n_channels, n_freqs)
        print("PSD FLAT shape:", psd_flat.shape)

        # Resolución frecuencial (mejor así que con sfreq/n_per_seg)
        freq_resolution = sfreq/n_per_seg
        print("Resolución frecuencial:", freq_resolution, "Hz")

        # Creamos FOOOFGroup para este sujeto
        fg_subject = FOOOFGroup(
            peak_width_limits=[2*freq_resolution, 12],
            max_n_peaks=4,
            min_peak_height=0.2,
            peak_threshold=2.0,
            aperiodic_mode=aperiodic_mode,   # 'fixed' o 'knee'
        )

        # Ajustamos el modelo para TODOS los espectros de este sujeto
        fg_subject.fit(freqs, psd_flat, n_jobs=25)
        print("FOOOF ajustado para el sujeto shaoe:", len(fg_subject))



        band_power_dictionary = {}
        band_ps_multiple=[]
        # Para cada banda definida:
        for label, definition in bands:
            
            # Array para guardar un valor por modelo
            # (power total sumado en esa banda)
            band_power = []
            i=0
            # Recorremos cada modelo FOOOF dentro del FOOOFGroup
            for f_res in fg_subject:
                i+=1
                # Extraemos TODOS los picos dentro de la banda
                band_ps = get_band_peak(
                    f_res.peak_params,
                    definition,
                    select_highest=select_highest
                )
                
                if select_highest==False:
                    # Si hay múltiples picos → sumamos el PW de todos
                    if band_ps.size > 3:
                        # print(f"mas de un pico en fooof {i} para la banda {label}")
                        total_power = np.sum(band_ps[:,1])   # columna 1 = PW
                    else:
                        
                        band_ps=check_nans(band_ps)
                        total_power = band_ps[1]  # columna 1 = PW
                        
                        
                elif select_highest==True:
                    band_ps=check_nans(band_ps)
                    total_power = band_ps[1]  # columna 1 = PW
                    

                band_power.append(total_power)

            # Convertimos a array
            band_power = np.array(band_power)

            # Guardamos en el diccionario
            band_power_dictionary[label] = band_power

        # Mostrar resumen
        for b, vals in band_power_dictionary.items():
            print(f"Banda {b}: {vals.shape} power values (1 valor por modelo)")

        aperiodic = fg_subject.get_params('aperiodic_params')   # array
        if aperiodic_mode=="knee":
            offsets, knees, exponents = aperiodic[:, 0], aperiodic[:, 1], aperiodic[:, 2]
        elif aperiodic_mode=="fixed":
            offsets, exponents = aperiodic[:, 0], aperiodic[:, 1]
            
        # Extraer R² del ajuste FOOOF
        r2 = fg_subject.get_params('r_squared')

        # Extraer error (RMSE del model fit)
        error = fg_subject.get_params('error')

            
            # Crear índices de epochs (0,1,2,... repetidos por canal)
                # Create relative and global epoch indices
        epoch_relative_idx = np.repeat(np.arange(n_epochs), n_channels)
        epoch_global_idx = np.repeat(epochs_cond.selection, n_channels)


            
        #append the fooof group to the dictionary
        dict_fg[cond_clean]=fg_subject

        
        # DataFrame del sujeto por condition
        df_fooof_subject_condition = pd.DataFrame({
            'Subject': [subject]* shape_tabla,
            'event_id': np.repeat(epoch_codes, n_channels),
            'Condition_self': np.repeat(Condition_self, n_channels),
            'Condition_emotion': np.repeat(Condition_emotion, n_channels),
            'Condition_gaze': np.repeat(Condition_gaze, n_channels),
            'Epoch': epoch_global_idx,
            'Epoch_relative': epoch_relative_idx,
            'Elect': np.tile(channels, n_epochs),
            
            # Band powers
            'delta':  band_power_dictionary.get('delta'),
            'theta':  band_power_dictionary.get('theta'),
            'alpha':  band_power_dictionary.get('alpha'),
            'beta':   band_power_dictionary.get('beta'),
            'gamma':  band_power_dictionary.get('gamma'),
            
            # Aperiodic
            'offsets': offsets,
            'exponents': exponents,
            
            # Metrics
            'r2': r2,
            'error': error
        })

        # Agregar knee en caso de ser necesario
        if aperiodic_mode == "knee":
            df_fooof_subject_condition['knee'] = knees
        list_df.append(df_fooof_subject_condition)
    
    df_fooof_subject = pd.concat(list_df, ignore_index=True)
        
    if return_fg_subject:
        return df_fooof_subject, dict_fg
    else:
        return df_fooof_subject





        

In [14]:
## tests

freq_bands = {
    'delta': (1, 4),
    'theta': (4, 8),
    'alpha': (8, 12),
    'beta': (12, 30),
    'gamma': (30, 40)
}  

bands = Bands(freq_bands)

aperiodic_mode="fixed" # "fixed" or "knee"

# fmin=1
# fmax=40

# select_channels="only_zinnen"

# condition="zinnen"
# condition_epoch=f"fix_{condition.upper()}"
return_fg_subject=True

select_channels=None
filtering=True
if filtering==True:
    lfreq=1
    hfreq=40
    filter_name= f"filt_{lfreq}-{hfreq}"
    print(f"Filtrado aplicado: {lfreq}-{hfreq} Hz")
elif filtering==False:
    print("No se ha aplicado filtrado.")

Filtrado aplicado: 1-40 Hz


In [15]:
# selected_subjects = [s for s in subjects if s not in subjects_remove]

In [16]:
df_fooof_subject_all = pd.DataFrame()
fg_subject_all={}

# df_fooof_subject_all_knee = pd.DataFrame()
# fg_subject_all_knee=[]

# for condition in conditions:
    # for h_subj in range(len(subjects)):
# for h_subj in range(1):  # probar con un solo sujeto primero

crop_epochs=None
all_tables = []
type_epoch="self"
if type_epoch=="emoc":
    crop_epochs=3
    # crop_epochs=None


for subject in subjects_BV:

    path_epochs= epochs_clean_path / f"{subject}_epochs_{type_epoch}-epo.fif"

    epochs = mne.read_epochs(path_epochs, preload=True)

    if crop_epochs is not None:
        epochs.crop(tmin=0, tmax=crop_epochs)
        print(f"Epochs cropped for {subject}: {epochs.tmin}-{crop_epochs} s")
    
    if filtering:
        epochs.filter(l_freq=lfreq, h_freq=hfreq, n_jobs=10)
        print(f"Filter applied to {subject}: {lfreq}-{hfreq} Hz")
        filter_applied=True
        

    # def compute_fooof_subject(epochs,condition, bands,dict_select_channels, aperiodic_mode="fixed", select_highest=False, select_channels=None,return_fg_subject=False ):

    df_fooof_subject, fg_subject= compute_fooof_subject(
        subject=subject,
        epochs=epochs,
        # contidion=condition, im removing it  to include it in the function
        bands=bands,
        
        aperiodic_mode=aperiodic_mode,
        change_name=None,
        select_highest=True,
        select_channels=select_channels,
        return_fg_subject=return_fg_subject
    )
    if return_fg_subject==True:
        fg_subject_all[subject] =fg_subject
        
    # df_fooof_subject_fixed, fg_subject_fixed= compute_fooof_subject(
    #     epochs_condition,
    #     condition,
    #     bands,
    #     dict_select_channels,
    #     aperiodic_mode="fixed",
    #     select_highest=False,
    #     select_channels=select_channels,
    #     return_fg_subject=return_fg_subject
    # )
    # if return_fg_subject==True:
    #     fg_subject_all_fixed.append(fg_subject_fixed)
    

    # Añadir al DataFrame general
    df_fooof_subject_all = pd.concat([df_fooof_subject_all, df_fooof_subject], ignore_index=True)


    del epochs, df_fooof_subject
    if return_fg_subject==True:
        del fg_subject

Reading g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\epochs_clean_event\s01b_epochs_self-epo.fif ...
    Found the data of interest:
        t =   -2000.00 ...       0.00 ms
        0 CTF compensation matrices available
Not setting metadata
213 matching events found
No baseline correction applied
0 projection items activated
Setting up band-pass filter from 1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 845 samples (3.301 s)



C:\Users\UCM\AppData\Local\Temp\ipykernel_9756\3957820985.py:30: RuntimeWarning: filter_length (845) is longer than the signal (513), distortion is likely. Reduce filter length or filter a longer signal.
  epochs.filter(l_freq=lfreq, h_freq=hfreq, n_jobs=10)
[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   8 tasks      | elapsed:    2.0s
[Parallel(n_jobs=10)]: Done 116 tasks      | elapsed:    2.0s
[Parallel(n_jobs=10)]: Done 10277 tasks      | elapsed:    2.9s
[Parallel(n_jobs=10)]: Done 12567 out of 12567 | elapsed:    3.1s finished


Filter applied to s01b: 1-40 Hz
Number of epochs: 7, Number of channels: 59,
Effective window size : 1.000 (s)


C:\Users\UCM\AppData\Local\Temp\ipykernel_9756\884320191.py:37: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data = epochs_cond.get_data()   # (n_epochs, n_channels, n_times)
[Parallel(n_jobs=20)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done   6 out of  20 | elapsed:    2.7s remaining:    6.5s
[Parallel(n_jobs=20)]: Done  11 out of  20 | elapsed:    2.8s remaining:    2.3s
[Parallel(n_jobs=20)]: Done  16 out of  20 | elapsed:    2.8s remaining:    0.6s
[Parallel(n_jobs=20)]: Done  20 out of  20 | elapsed:    2.9s finished


PSD FLAT shape: (413, 40)
Resolución frecuencial: 0.3333333333333333 Hz
Running FOOOFGroup across 413 power spectra.
FOOOF ajustado para el sujeto shaoe: 413
Banda delta: (413,) power values (1 valor por modelo)
Banda theta: (413,) power values (1 valor por modelo)
Banda alpha: (413,) power values (1 valor por modelo)
Banda beta: (413,) power values (1 valor por modelo)
Banda gamma: (413,) power values (1 valor por modelo)
Number of epochs: 9, Number of channels: 59,
Effective window size : 1.000 (s)
PSD FLAT shape: (531, 40)
Resolución frecuencial: 0.3333333333333333 Hz
Running FOOOFGroup across 531 power spectra.


...

FOOOF ajustado para el sujeto shaoe: 295
Banda delta: (295,) power values (1 valor por modelo)
Banda theta: (295,) power values (1 valor por modelo)
Banda alpha: (295,) power values (1 valor por modelo)
Banda beta: (295,) power values (1 valor por modelo)
Banda gamma: (295,) power values (1 valor por modelo)
Number of epochs: 9, Number of channels: 59,
Effective window size : 1.000 (s)
PSD FLAT shape: (531, 40)
Resolución frecuencial: 0.3333333333333333 Hz
Running FOOOFGroup across 531 power spectra.


C:\Users\UCM\AppData\Local\Temp\ipykernel_9756\884320191.py:37: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data = epochs_cond.get_data()   # (n_epochs, n_channels, n_times)
[Parallel(n_jobs=20)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done   6 out of  20 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=20)]: Done  11 out of  20 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=20)]: Done  16 out of  20 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=20)]: Done  20 out of  20 | elapsed:    0.0s finished


FOOOF ajustado para el sujeto shaoe: 531
Banda delta: (531,) power values (1 valor por modelo)
Banda theta: (531,) power values (1 valor por modelo)
Banda alpha: (531,) power values (1 valor por modelo)
Banda beta: (531,) power values (1 valor por modelo)
Banda gamma: (531,) power values (1 valor por modelo)
Number of epochs: 6, Number of channels: 59,
Effective window size : 1.000 (s)
PSD FLAT shape: (354, 40)
Resolución frecuencial: 0.3333333333333333 Hz
Running FOOOFGroup across 354 power spectra.


C:\Users\UCM\AppData\Local\Temp\ipykernel_9756\884320191.py:37: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data = epochs_cond.get_data()   # (n_epochs, n_channels, n_times)
[Parallel(n_jobs=20)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done   6 out of  20 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=20)]: Done  11 out of  20 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=20)]: Done  16 out of  20 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=20)]: Done  20 out of  20 | elapsed:    0.0s finished


FOOOF ajustado para el sujeto shaoe: 354
Banda delta: (354,) power values (1 valor por modelo)
Banda theta: (354,) power values (1 valor por modelo)
Banda alpha: (354,) power values (1 valor por modelo)
Banda beta: (354,) power values (1 valor por modelo)
Banda gamma: (354,) power values (1 valor por modelo)


In [17]:
# if select_channels:
#     if filtering==True and filter_applied==True:
#         df_fooof_subject_all.to_pickle(ACW_path / f"df_fooof_subject_all_{filter_name}_{aperiodic_mode}_{select_channels}_{layer_script}.pickle")
#         fg_subject_all_path=ACW_path / f"fg_subject_all_{filter_name}_{aperiodic_mode}_{select_channels}_{layer_script}.pkl"
#         print(f"Saved df_fooof_subject_all_{filter_name}_{aperiodic_mode}_{select_channels}_{layer_script}.pickle")
#     else:
#         df_fooof_subject_all.to_pickle(ACW_path / f"df_fooof_subject_all_{aperiodic_mode}_{select_channels}_{layer_script}.pickle")
#         fg_subject_all_path=ACW_path / f"fg_subject_all_{aperiodic_mode}_{select_channels}_{layer_script}.pkl"
#         print(f"Saved df_fooof_subject_all_{aperiodic_mode}_{select_channels}_{layer_script}.pickle")
# else:
#     if filtering==True and filter_applied==True:
#         df_fooof_subject_all.to_pickle(ACW_path / f"df_fooof_subject_all_{filter_name}_{aperiodic_mode}_{layer_script}.pickle")
#         fg_subject_all_path=ACW_path / f"fg_subject_all_{filter_name}_{aperiodic_mode}_{layer_script}.pkl"
#         print(f"Saved df_fooof_subject_all_{filter_name}_{aperiodic_mode}_{layer_script}.pickle")
#     else:   
#         df_fooof_subject_all.to_pickle(ACW_path / f"df_fooof_subject_all_{aperiodic_mode}_{layer_script}.pickle")
#         fg_subject_all_path=ACW_path / f"fg_subject_all_{aperiodic_mode}_{layer_script}.pkl"
#         print(f"Saved df_fooof_subject_all_{aperiodic_mode}_{layer_script}.pickle")

In [18]:
df_fooof_subject_all["Subject"].unique

<bound method Series.unique of 0         s01b
1         s01b
2         s01b
3         s01b
4         s01b
          ... 
453941    s36b
453942    s36b
453943    s36b
453944    s36b
453945    s36b
Name: Subject, Length: 453946, dtype: object>

In [19]:
# -------------------------------
# Construcción del sufijo del nombre
# -------------------------------
suffix = []
suffix.append(type_epoch)

if filtering and filter_applied:
    suffix.append(filter_name)

suffix.append(aperiodic_mode)

if select_channels:
    suffix.append(select_channels)

suffix.append(layer_script)

if crop_epochs is not None:
    suffix.append(f"crop_{crop_epochs}")

suffix_str = "_".join(suffix)

# -------------------------------
# Rutas de guardado
# -------------------------------
df_path = ACW_path / f"df_fooof_subject_all_{suffix_str}.pickle"
fg_subject_all_path = ACW_path / f"fg_subject_all_{suffix_str}.pkl"

# -------------------------------
# Guardar DataFrame
# -------------------------------
df_fooof_subject_all.to_pickle(df_path)
print(f"Saved {df_path.name}")

# -------------------------------
# Guardar FOOOF groups (diccionario anidado)
# -------------------------------
with open(fg_subject_all_path, "wb") as f:
    pickle.dump(fg_subject_all, f)

print(f"Saved {fg_subject_all_path.name}")

Saved df_fooof_subject_all_self_filt_1-40_fixed_event.pickle
Saved fg_subject_all_self_filt_1-40_fixed_event.pkl


In [20]:
# import random
# import matplotlib.pyplot as plt
# from fooof import FOOOFGroup

# # =====================================================
# # 1) COMBINAR TODOS LOS FOOOFGroup EN UNO SOLO
# # =====================================================
# def merge_fooofgroups(fg_list):
#     """Une múltiples FOOOFGroup en uno solo."""
    
#     # Crear un FOOOFGroup vacío
#     merged = FOOOFGroup()
    
#     # Recuperar freqs del primero
#     merged.freqs = fg_list[0].freqs
    
#     # Concatenar todos los power spectra
#     merged.power_spectra = np.vstack([fg.power_spectra for fg in fg_list])
    
#     # Combinar los resultados
#     merged.group_results = []
#     for fg in fg_list:
#         merged.group_results.extend(fg.group_results)

#     return merged

# fg_merged = merge_fooofgroups(fg_subject_all)

# import random
# import matplotlib.pyplot as plt

# # 1) Número total de modelos
# n_available = len(fg_merged)
# n_plots = min(50, n_available)

# # 2) Seleccionar 50 modelos aleatorios
# selected_indices = random.sample(range(n_available), n_plots)

# print(f"Guardanddo {n_plots} modelos de {n_available} disponibles.")

# # =============================================================
# #   CREAR CARPETAS
# # =============================================================

# root_path = ACW_path / f"plot_FOOOF_{select_channels}_{aperiodic_mode}_{layer_script}"
# root_path.mkdir(parents=True, exist_ok=True)

# save_path_linear = root_path / f"plot_FOOOF_only_zinnen_{layer_script}_linear_power"
# save_path_linear.mkdir(parents=True, exist_ok=True)

# save_path_log = root_path / f"plot_FOOOF_only_zinnen_{layer_script}_log_power"
# save_path_log.mkdir(parents=True, exist_ok=True)

# # =============================================================
# #   EXTRAER, PLOTEAR Y GUARDAR
# # =============================================================

# for idx in selected_indices:

#     # EXTRAER MODELO INDIVIDUAL
#     fm = fg_merged.get_fooof(idx)

#     # ============================
#     #   PLOT 1: LINEAR
#     # ============================
#     fig_lin, ax_lin = plt.subplots()
#     fm.plot(ax=ax_lin,plot_peaks="shade-dot", plt_log=False)
#     fname_lin = f"fooof_model_{idx:04d}_plot_linear.png"
#     fig_lin.savefig(save_path_linear / fname_lin, dpi=200, bbox_inches="tight")
#     plt.close(fig_lin)

#     # ============================
#     #   PLOT 2: LOG
#     # ============================
#     fig_log, ax_log = plt.subplots()
#     fm.plot(ax=ax_log,plot_peaks="shade-dot", plt_log=True)
#     fname_log = f"fooof_model_{idx:04d}_plot_log.png"
#     fig_log.savefig(save_path_log / fname_log, dpi=200, bbox_inches="tight")
#     plt.close(fig_log)

# print(f"""
# Listo: {n_plots * 2} figuras guardadas en:

# - {save_path_linear}
# - {save_path_log}
# """)


In [21]:
# =====================================================
#   PLOTEAR Y GUARDAR FIGURAS
# =====================================================

# n_available = len(fg_merged)
# n_plots = min(50, n_available)
# selected_indices = random.sample(range(n_available), n_plots)

In [22]:
# import random
# import matplotlib.pyplot as plt
# from fooof import FOOOFGroup

# # =====================================================
# # 1) COMBINAR TODOS LOS FOOOFGroup EN UNO SOLO
# # =====================================================
# def merge_fooofgroups(fg_list):
#     """Une múltiples FOOOFGroup en uno solo."""
    
#     merged = FOOOFGroup()
#     merged.freqs = fg_list[0].freqs
#     merged.power_spectra = np.vstack([fg.power_spectra for fg in fg_list])
    
#     merged.group_results = []
#     for fg in fg_list:
#         merged.group_results.extend(fg.group_results)

#     return merged

# # =====================================================
# #   MERGE DE FG SEGÚN MODO (KNEE O FIXED)
# # =====================================================

# if aperiodic_mode == "fixed":
#     fg_merged = merge_fooofgroups(fg_subject_all_fixed)
# elif aperiodic_mode == "knee":
#     fg_merged = merge_fooofgroups(fg_subject_all_knee)
# else:
#     raise ValueError("aperiodic_mode debe ser 'fixed' o 'knee'")



# print(f"Guardando {n_plots} modelos de {n_available} para modo: {aperiodic_mode}")

#     # =====================================================
# #   CREAR CARPETAS SEGÚN MODO (fixed / knee)
# # =====================================================

# root_path = ACW_path / f"plot_FOOOF_{select_channels}_{layer_script}"
# root_path.mkdir(parents=True, exist_ok=True)

# save_path_linear = root_path / f"plot_FOOOF_{select_channels}_{aperiodic_mode}_{layer_script}_linear_power"
# save_path_linear.mkdir(parents=True, exist_ok=True)

# save_path_log = root_path / f"plot_FOOOF_{select_channels}_{aperiodic_mode}_{layer_script}_log_power"
# save_path_log.mkdir(parents=True, exist_ok=True)

# # =====================================================
# #   LOOP DE PLOTS
# # =====================================================

# for idx in selected_indices:

#     fm = fg_merged.get_fooof(idx)

#     # ----- LINEAR -----
#     fig_lin, ax_lin = plt.subplots()
#     fm.plot(ax=ax_lin, plot_peaks="shade-dot", plt_log=False)
#     fname_lin = f"fooof_model_{idx:04d}_{select_channels}_{aperiodic_mode}_{layer_script}_plot_linear.png"
#     fig_lin.savefig(save_path_linear / fname_lin, dpi=200, bbox_inches="tight")
#     plt.close(fig_lin)

#     # ----- LOG -----
#     fig_log, ax_log = plt.subplots()
#     fm.plot(ax=ax_log, plot_peaks="shade-dot", plt_log=True)
#     fname_log = f"fooof_model_{idx:04d}_{select_channels}_{aperiodic_mode}_{layer_script}_plot_log.png"
#     fig_log.savefig(save_path_log / fname_log, dpi=200, bbox_inches="tight")
#     plt.close(fig_log)

# print(f"""
# Listo: {n_plots * 2} figuras guardadas en:

# - {save_path_linear}
# - {save_path_log}
# """)
